In [ ]:
import numpy as np 
import pandas as pd 
import os
import scipy.io as sio
from PIL import Image

In [ ]:
def load_pascal3d_dataset(base_path, class_name, dataset_source='imagenet', subset='train'):
    """
    Loads images and annotations from the PASCAL3D+ 1.1 dataset.
    
    Args:
        base_path (str): Root directory of the dataset.
        class_name (str): Object category (e.g., 'aeroplane', 'bus').
        dataset_source (str): 'imagenet' or 'pascal'.
        subset (str): Subset name (e.g., 'train', 'val').
        
    Returns:
        List of dictionaries containing image ID, path, PIL Image, and the parsed annotation.
    """
    # Define paths based on the PASCAL3D+ directory structure
    images_dir = os.path.join(base_path, 'Images', f"{class_name}_{dataset_source}")
    annotations_dir = os.path.join(base_path, 'Annotations', f"{class_name}_{dataset_source}")
    split_file = os.path.join(base_path, 'Image_sets', f"{class_name}_{dataset_source}_{subset}.txt")
    
    # Ensure the Image_sets file exists before proceeding
    if not os.path.exists(split_file):
        raise FileNotFoundError(f"Split file not found: {split_file}")
        
    # Extract image IDs from the text file
    with open(split_file, 'r') as f:
        image_ids = [line.strip() for line in f.readlines() if line.strip()]
        
    dataset = []
    
    for img_id in image_ids:
        # ImageNet uses .JPEG, PASCAL VOC uses .jpg
        ext = '.JPEG' if dataset_source == 'imagenet' else '.jpg'
        img_path = os.path.join(images_dir, f"{img_id}{ext}")
        
        # Fallback check just in case the extension differs
        if not os.path.exists(img_path):
            ext = '.jpg' if ext == '.JPEG' else '.JPEG'
            img_path = os.path.join(images_dir, f"{img_id}{ext}")
            
        anno_path = os.path.join(annotations_dir, f"{img_id}.mat")
        
        # Verify both the image and the annotation exist
        if os.path.exists(img_path) and os.path.exists(anno_path):
            
            # Load MATLAB annotation
            # squeeze_me & struct_as_record convert MATLAB structs into Python objects
            mat_data = sio.loadmat(anno_path, squeeze_me=True, struct_as_record=False)
            
            # The root of the PASCAL3D+ annotation is stored under the 'record' key
            annotation = mat_data['record']
            
            # Lazy-load the image using PIL (saves memory until processing)
            image = Image.open(img_path) 
            
            dataset.append({
                'image_id': img_id,
                'image_path': img_path,
                'image': image,
                'annotation': annotation
            })
        else:
            print(f"Warning: Missing image or annotation for ID {img_id}")
            
    return dataset

# PASCAL3D+ Data Parsing

In [ ]:
# The 12 classes in PASCAL3D+
CLASSES = [
    'aeroplane', 'bicycle', 'boat', 'bottle', 'bus', 'car', 
    'chair', 'diningtable', 'motorbike', 'sofa', 'train', 'tvmonitor'
]

def parse_voc_split(txt_path):
    """Parses PASCAL VOC Main txt files and returns IDs of positive/difficult instances."""
    valid_ids = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            # PASCAL VOC format: <image_id> <flag>
            # 1 = present, 0 = difficult/truncated, -1 = absent
            if len(parts) >= 2 and parts[-1] in ['1', '0']:
                valid_ids.append(parts[0])
    return valid_ids

def parse_imagenet_split(txt_path):
    """Parses simple line-by-line ImageNet txt files."""
    with open(txt_path, 'r') as f:
        return [line.strip() for line in f.readlines() if line.strip()]

def build_dataset_splits(base_path):
    """
    Builds lists of (image_path, annotation_path, class_name) for Train, Val, and Test.
    """
    splits = {
        'train': [], # ImageNet Train + PASCAL Train
        'val': [],   # ImageNet Val
        'test': []   # PASCAL Val
    }
    
    for cls in CLASSES:
        # Paths
        img_net_dir = os.path.join(base_path, 'Images', f"{cls}_imagenet")
        img_pas_dir = os.path.join(base_path, 'Images', f"{cls}_pascal")
        
        ann_net_dir = os.path.join(base_path, 'Annotations', f"{cls}_imagenet")
        ann_pas_dir = os.path.join(base_path, 'Annotations', f"{cls}_pascal")
        
        # ImageNet Train & Val
        for split_name in ['train', 'val']:
            txt_path = os.path.join(base_path, 'Image_sets', f"{cls}_imagenet_{split_name}.txt")
            if os.path.exists(txt_path):
                img_ids = parse_imagenet_split(txt_path)
                for img_id in img_ids:
                    # ImageNet typically uses .JPEG
                    img_path = os.path.join(img_net_dir, f"{img_id}.JPEG")
                    ann_path = os.path.join(ann_net_dir, f"{img_id}.mat")
                    
                    if os.path.exists(img_path) and os.path.exists(ann_path):
                        splits[split_name].append((img_path, ann_path, cls))

        # PASCAL Train & Test (Val)
        voc_train_txt = os.path.join(base_path, 'PASCAL', 'VOCdevkit', 'VOC2012', 'ImageSets', 'Main', f"{cls}_train.txt")
        voc_val_txt = os.path.join(base_path, 'PASCAL', 'VOCdevkit', 'VOC2012', 'ImageSets', 'Main', f"{cls}_val.txt")
        
        # PASCAL Train goes to our generic 'train' split
        if os.path.exists(voc_train_txt):
            train_ids = parse_voc_split(voc_train_txt)
            for img_id in train_ids:
                img_path = os.path.join(img_pas_dir, f"{img_id}.jpg")
                ann_path = os.path.join(ann_pas_dir, f"{img_id}.mat")
                # Ensure PASCAL3D+ actually annotated this VOC image
                if os.path.exists(img_path) and os.path.exists(ann_path):
                    splits['train'].append((img_path, ann_path, cls))
                    
        # PASCAL Val goes to our generic 'test' split
        if os.path.exists(voc_val_txt):
            test_ids = parse_voc_split(voc_val_txt)
            for img_id in test_ids:
                img_path = os.path.join(img_pas_dir, f"{img_id}.jpg")
                ann_path = os.path.join(ann_pas_dir, f"{img_id}.mat")
                if os.path.exists(img_path) and os.path.exists(ann_path):
                    splits['test'].append((img_path, ann_path, cls))
                    
    return splits['train'], splits['val'], splits['test']


In [ ]:
import math
import torch
from torch.utils.data import Dataset
from torchvision import transforms


class Pascal3DAzimuthDataset(Dataset):
    def __init__(self, data_split, target_class, transform=None):
        self.transform = transform
        self.samples = [] 
        
        print(f"Parsing annotations to build object index for '{target_class}'...")
        
        for img_path, ann_path, cls in data_split:
            if cls != target_class:
                continue
                
            mat_data = sio.loadmat(ann_path, squeeze_me=True, struct_as_record=False)
            objects = mat_data['record'].objects
            
            if not isinstance(objects, np.ndarray):
                objects = [objects]
                
            for obj in objects:
                obj_class = getattr(obj, 'class', None)
                
                if obj_class == target_class and hasattr(obj, 'viewpoint'):
                    azimuth_deg = getattr(obj.viewpoint, 'azimuth', None)
                    
                    if azimuth_deg is not None:
                        azimuth_rad = azimuth_deg * (math.pi / 180.0)
                        
                        self.samples.append({
                            'img_path': img_path,
                            'bbox': obj.bbox, 
                            'azimuth_rad': azimuth_rad
                        })
                        
        print(f"Done! Found {len(self.samples)} valid {target_class} objects.")
        
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Load the full image
        image = Image.open(sample['img_path']).convert("RGB")
        
        # Crop the image to isolate the object using the bounding box
        bbox = sample['bbox']
        # Pillow crop expects (left, upper, right, lower)
        crop = image.crop((bbox[0], bbox[1], bbox[2], bbox[3]))
        
        # Apply standard transformations (e.g., Resize to 224x224, ToTensor)
        if self.transform:
            crop = self.transform(crop)
            
        # Target must be a float tensor for regression tasks
        target = torch.tensor(sample['azimuth_rad'], dtype=torch.float32)
        
        return crop, target

In [ ]:
PASCAL3D_ROOT = "/kaggle/input/datasets/rajdeeppathak/pascal3d/PASCAL3D+_release1.1"
    
print("Parsing directories and building splits...")
train_list, val_list, test_list = build_dataset_splits(PASCAL3D_ROOT)

print(f"Found Train samples: {len(train_list)}")
print(f"Found Validation samples (ImageNet Val): {len(val_list)}")
print(f"Found Test samples (PASCAL Val): {len(test_list)}")

In [ ]:
# Saving the splits to load it later, because the above cell takes some time to run.

import json
with open("train_list.json", "w") as f:
    json.dump(train_list, f, indent=4)

with open("val_list.json", "w") as f:
    json.dump(val_list, f, indent=4)

with open("test_list.json", "w") as f:
    json.dump(test_list, f, indent=4)

In [ ]:
# Loading the splits if saved previously

import json

with open("/kaggle/working/train_list.json", "r") as file:
    train_list = json.load(file)

with open("/kaggle/working/val_list.json", "r") as file:
    val_list = json.load(file)

with open("/kaggle/working/test_list.json", "r") as file:
    test_list = json.load(file)

# ConvNeXt: Fine-Tuning and Feature Extraction

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import math
import numpy as np
import scipy.io as sio
from PIL import Image
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ConvNeXt standard resolution is 224x224
convnext_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Define the Multi-Class Biternion Head
class MultiClassBiternionHead(nn.Module):
    def __init__(self, in_features, num_classes=12):
        super().__init__()
        self.num_classes = num_classes
        self.linear = nn.Linear(in_features, num_classes * 2)
        
    def forward(self, x):
        x = self.linear(x)
        x = x.view(-1, self.num_classes, 2)
        return nn.functional.normalize(x, p=2, dim=2)

# Initialize ConvNeXt (Using Tiny variant as standard baseline)
print("Initializing ConvNeXt Tiny...")
global_convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

# ConvNeXt's classifier is a Sequential block. The linear layer is at index 2.
# We read its in_features (768 for Tiny) and replace just that layer.
in_features = global_convnext.classifier[2].in_features
global_convnext.classifier[2] = MultiClassBiternionHead(in_features=in_features, num_classes=12)
global_convnext = global_convnext.to(device)

# Define a fixed mapping for the 12 PASCAL3D+ classes
CLASSES = [
    'aeroplane', 'bicycle', 'boat', 'bottle', 'bus', 'car', 
    'chair', 'diningtable', 'motorbike', 'sofa', 'train', 'tvmonitor'
]
CLASS_TO_IDX = {cls_name: i for i, cls_name in enumerate(CLASSES)}

class Pascal3DGlobalDataset(Dataset):
    def __init__(self, data_split, transform=None):
        self.transform = transform
        self.samples = [] 
        
        print("Parsing annotations to build global object index...")
        
        for img_path, ann_path, cls in data_split:
            try:
                mat_data = sio.loadmat(ann_path, squeeze_me=True, struct_as_record=False)
                if not hasattr(mat_data['record'], 'objects'):
                    continue
                    
                objects = mat_data['record'].objects
                
                if not isinstance(objects, np.ndarray):
                    objects = [objects]
                    
                for obj in objects:
                    obj_class = getattr(obj, 'class', None)
                    
                    if obj_class in CLASS_TO_IDX and hasattr(obj, 'viewpoint'):
                        azimuth_deg = getattr(obj.viewpoint, 'azimuth', None)
                        
                        if azimuth_deg is not None:
                            azimuth_rad = azimuth_deg * (math.pi / 180.0)
                            self.samples.append({
                                'img_path': img_path,
                                'bbox': obj.bbox,
                                'azimuth_rad': azimuth_rad,
                                'class_idx': CLASS_TO_IDX[obj_class]
                            })
            except Exception:
                pass
                        
        print(f"Done! Found {len(self.samples)} valid objects across all classes.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample['img_path']).convert("RGB")
        
        bbox = sample['bbox']
        # Cast to int to prevent PIL float errors
        crop = image.crop((int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])))
        
        if self.transform:
            crop = self.transform(crop)
            
        target = torch.tensor(sample['azimuth_rad'], dtype=torch.float32)
        class_idx = torch.tensor(sample['class_idx'], dtype=torch.long)
        
        return crop, target, class_idx

In [ ]:
# Create the dataloaders
# train_list and val_list must be loaded in memory
global_train_dataset = Pascal3DGlobalDataset(train_list, transform=convnext_transforms)
train_loader = DataLoader(global_train_dataset, batch_size=64, shuffle=True, num_workers=2)

global_val_dataset = Pascal3DGlobalDataset(val_list, transform=convnext_transforms)
val_loader = DataLoader(global_val_dataset, batch_size=128, shuffle=False, num_workers=2)

optimizer = AdamW(global_convnext.parameters(), lr=1e-4, weight_decay=0.01)

epochs = 20
best_val_loss = float('inf') 

print("Starting Global Fine-tuning on ConvNeXt...")

for epoch in range(epochs):    
    # Training phase
    global_convnext.train() 
    total_train_loss = 0
    
    for images, targets_rad, class_idx in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False):
        images = images.to(device)
        targets_rad = targets_rad.to(device)
        class_idx = class_idx.to(device)
        
        target_biternion = torch.stack([torch.cos(targets_rad), torch.sin(targets_rad)], dim=1)
        
        optimizer.zero_grad()
        
        all_class_preds = global_convnext(images)
        
        batch_size = images.size(0)
        batch_indices = torch.arange(batch_size)
        relevant_preds = all_class_preds[batch_indices, class_idx] 
        
        dot_product = torch.sum(relevant_preds * target_biternion, dim=1)
        loss = torch.mean(1.0 - dot_product)
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation phase
    global_convnext.eval() 
    total_val_loss = 0
    
    with torch.no_grad():
        for images, targets_rad, class_idx in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False):
            images = images.to(device)
            targets_rad = targets_rad.to(device)
            class_idx = class_idx.to(device)
            
            target_biternion = torch.stack([torch.cos(targets_rad), torch.sin(targets_rad)], dim=1)
            
            all_class_preds = global_convnext(images)
            
            batch_size = images.size(0)
            batch_indices = torch.arange(batch_size)
            relevant_preds = all_class_preds[batch_indices, class_idx] 
            
            dot_product = torch.sum(relevant_preds * target_biternion, dim=1)
            loss = torch.mean(1.0 - dot_product)
            
            total_val_loss += loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader)
    
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # Save best model based on validation
    if avg_val_loss < best_val_loss:
        print(f"  --> Validation loss improved from {best_val_loss:.4f} to {avg_val_loss:.4f}. Saving best model!")
        best_val_loss = avg_val_loss
        torch.save(global_convnext.state_dict(), "global_convnext_tiny_best.pth")

# Save final model
torch.save(global_convnext.state_dict(), "global_convnext_tiny_final.pth")
print("\nTraining Complete!")
print("Best weights saved as: 'global_convnext_tiny_best.pth'")
print("Final weights saved as: 'global_convnext_tiny_final.pth'")

In [ ]:
def extract_features(dataset, model, device, batch_size=32):
    """
    Passes a dataset through a model and returns the features and targets as tensors.
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    all_features = []
    all_targets = []
    
    # Disable gradient calculation for faster inference and lower memory usage
    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Extracting Features"):
            images = images.to(device)
            
            # Extract features (Output shape: [Batch_Size, 768])
            features = model(images)
            
            # Move features back to CPU to prevent GPU memory overflow if the dataset is very large
            all_features.append(features.cpu())
            all_targets.append(targets.cpu())
            
    # Concatenate lists into single tensors
    X = torch.cat(all_features, dim=0)
    theta = torch.cat(all_targets, dim=0)
    
    return X, theta

In [ ]:
'''
Re-run this cell and the next by changing TARGET to CLASSES[1], CLASSES[2], ..., CLASSES[11] to extract features and train ANGLE for all 12 classes.
'''

from tqdm.auto import tqdm
import torch.nn as nn
import torchvision.models as models

try:
    print("Loading saved ConvNeXt Tiny weights...")
    global_convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    # Setup the head to match the architecture of the saved weights
    in_features = global_convnext.classifier[2].in_features
    global_convnext.classifier[2] = MultiClassBiternionHead(in_features=in_features, num_classes=12)
    global_convnext = global_convnext.to(device)
    # Load the best fine-tuned weights
    state_dict = torch.load("/kaggle/working/global_convnext_tiny_best.pth", map_location=device, weights_only=True)
    global_convnext.load_state_dict(state_dict)
except:
    print("Error loading saved model weights. Make sure to fine-tune ConvNeXt first.")
    
TARGET = CLASSES[0] 

# Turn ConvNeXt into a feature extractor
# We preserve the Flatten (classifier[0]) and LayerNorm (classifier[1])
# and only replace the final Linear layer (classifier[2]) with Identity.
global_convnext.classifier[2] = nn.Identity()
global_convnext.eval() # Freezes BatchNorm/Dropout

# Prepare class-specific dataset
train_dataset = Pascal3DAzimuthDataset(
    data_split=train_list, 
    target_class=TARGET, 
    transform=convnext_transforms
)

val_dataset = Pascal3DAzimuthDataset(
    data_split=val_list, 
    target_class=TARGET, 
    transform=convnext_transforms
)

test_dataset = Pascal3DAzimuthDataset(
    data_split=test_list, 
    target_class=TARGET, 
    transform=convnext_transforms
)

# Extract Features using the helper function
print(f"Extracting ConvNeXt Tiny features for {TARGET}...")
X_train, theta_train = extract_features(train_dataset, global_convnext, device)
X_test, theta_test = extract_features(test_dataset, global_convnext, device)
X_val, theta_val = extract_features(val_dataset, global_convnext, device)

# Ensure theta_train is correctly shaped
theta_train = theta_train.unsqueeze(1)

# For ConvNeXt Tiny, the output shape will be (N, 768)
print(f"Features Extracted! X_train Shape: {X_train.shape}")

In [ ]:
from anglepy import ANGLE
from anglepy.metrics import mean_absolute_angular_deviation, circular_mean_directional_error, mean_circular_crps, accuracy, med_err
from tqdm.auto import tqdm

best_params_path = f"/kaggle/working/Hyperparameters/best_params_{TARGET}.json"

with open(best_params_path, "r") as f:
    loaded_params = json.load(f)
maads, crpss, cmdes, accuracies, mederrs = [], [], [], [], []

for _ in tqdm(range(50)):
    # Train the model by unpacking the dictionary
    model = ANGLE(
        X_train, theta_train, 
        **loaded_params, # Automatically inserts num_layer, hidden_dim, lr, etc.
        num_epochs=500, 
        print_every_nepoch=1000, 
        print_times_per_epoch=1, 
        batch_size=None, 
        standardize=False, 
        device=device,
        verbose=False, 
        circular_indices=None
    )
    
    print("Predicting...")
    y_pred = model.predict(X_test)
    y_pred_ensemble = model.sample(X_test, sample_size=100)
    maad_mean = mean_absolute_angular_deviation(y_pred, theta_test, return_degrees=True).item()
    cmde = circular_mean_directional_error(y_pred, theta_test).item()
    crps = mean_circular_crps(y_pred_ensemble, theta_test.reshape(-1,1), return_degrees=True)[0]
    acc = accuracy(y_pred, theta_test, theta=math.pi/6)
    median_err = med_err(y_pred, theta_test)
    accuracies.append(round(acc,3))
    mederrs.append(round(median_err,3))
    crpss.append(round(crps, 3))
    maads.append(round(maad_mean, 3))
    cmdes.append(round(cmde,3))
print(f"Acc: {np.mean(accuracies):.2f}\pm{np.std(accuracies):.2f}")
print(f"MedErr: {np.mean(mederrs):.2f}\pm{np.std(mederrs):.2f}")
print(f"CRPS: {np.mean(crpss):.2f}\pm{np.std(crpss):.2f}")
print(f"MAAD: {np.mean(maads):.2f}\pm{np.std(maads):.2f}")
print(f"CMDE: {np.mean(cmdes):.2f}\pm{np.std(cmdes):.2f}")